We want to take a real dataset from TabZilla and run models we already have

In [1]:
import sys
import openml
import sklearn
import xgboost

print("Python:", sys.executable)
print("OpenML:", openml.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

Python: /home/eng/gaidaya/projects/tabular-data-benchmarks/venv/bin/python
OpenML: 0.15.1
Scikit-learn: 1.6.1
XGBoost: 2.1.4


In [2]:
# Official TabZilla benchmark suite on OpenML
suite = openml.study.get_suite(379)

task_ids = suite.tasks

print("Number of tasks:", len(task_ids))
print("First 10 task IDs:", task_ids[:10])

Number of tasks: 36
First 10 task IDs: [7, 10, 11, 14, 22, 25, 29, 31, 45, 50]


Every task is defiening:
Dataset – על איזה נתונים עובדים.

Target – איזו עמודה מנסים לחזות.

Train/Test Splits – כיצד מחלקים את הנתונים לצורך הערכה.

Evaluation – כיצד מודדים את ביצועי המודל.

Every task is a dateset that checking a model preformance on types of "problems". 

In [3]:
# Inspect the first task in the TabZilla benchmark
task_id = task_ids[0]

task = openml.tasks.get_task(task_id)
dataset = task.get_dataset()

print("Task ID:", task_id)
print("Dataset name:", dataset.name)
print("Number of rows:", dataset.qualities.get("NumberOfInstances"))
print("Number of features:", dataset.qualities.get("NumberOfFeatures"))
print("Target column:", task.target_name)
print("Task type:", task.task_type)

Task ID: 7
Dataset name: audiology
Number of rows: 226.0
Number of features: 70.0
Target column: class
Task type: Supervised Classification


The dataset above is from the area of audio checking. 
that's a small dataset so we will try it first on our models. 
we fisrt need to check if the dataset is including strings as categorical_indicator (becuase LR for example is working only with numbers). 
so we will want to know if we need to change something. 

In [4]:
# Load the dataset as a pandas DataFrame
X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=task.target_name,
    dataset_format="dataframe"
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature types:")
print(X.dtypes.value_counts())

print("\nMissing values:", X.isna().sum().sum())
print("Number of classes:", y.nunique())

display(X.head())

X shape: (226, 69)
y shape: (226,)

Feature types:
category    60
category     4
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

Missing values: 317
Number of classes: 24


,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,...,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged
0,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
1,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
2,t,normal,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,good,t,a,f,f,f
3,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,unmeasured,t,a,f,f,f
4,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,...,normal,f,f,f,normal,t,a,f,f,f


we see that all the vallues are not numbers so we can not just let them throgh LR.
we will classifate them to be numbers. 

In [8]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Use the official OpenML task split
train_idx, test_idx = task.get_train_test_split_indices(
    repeat=0, fold=0, sample=0
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# All features in Audiology are categorical
categorical_cols = X.columns.tolist()

preprocessing = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent"
        )),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]), categorical_cols)
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000
    ),
    "Logistic Regression + L1": LogisticRegression(
        penalty="l1",
        solver="saga",
        C=10,
        max_iter=3000,
        random_state=42
    )
}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    print(f"{name}: Accuracy = {accuracy:.3f}")

Logistic Regression: Accuracy = 0.826
Logistic Regression + L1: Accuracy = 0.913


In [7]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

for C in [0.01, 0.1, 1, 10, 100]:
    model = Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("classifier", LogisticRegression(
            penalty="l1",
            solver="saga",
            C=C,
            max_iter=10000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    weights = model.named_steps["classifier"].coef_
    nonzero = (weights != 0).sum()

    print(
        f"C={C:<6} | Train={train_acc:.3f} | "
        f"Test={test_acc:.3f} | Nonzero weights={nonzero}"
    )

C=0.01   | Train=0.251 | Test=0.261 | Nonzero weights=0
C=0.1    | Train=0.463 | Test=0.478 | Nonzero weights=11
C=1      | Train=0.892 | Test=0.826 | Nonzero weights=123
C=10     | Train=1.000 | Test=0.913 | Nonzero weights=297
C=100    | Train=1.000 | Test=0.913 | Nonzero weights=855


In [9]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# XGBoost expects class labels encoded as integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Build the XGBoost pipeline
xgb_pipeline = Pipeline([
    ("preprocessing", clone(preprocessing)),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=2
    ))
])

# Train
xgb_pipeline.fit(X_train, y_train_encoded)

# Evaluate
predictions = xgb_pipeline.predict(X_test)

train_acc = accuracy_score(
    y_train_encoded,
    xgb_pipeline.predict(X_train)
)
test_acc = accuracy_score(y_test_encoded, predictions)
balanced_acc = balanced_accuracy_score(y_test_encoded, predictions)

print(f"XGBoost Train Accuracy: {train_acc:.3f}")
print(f"XGBoost Test Accuracy: {test_acc:.3f}")
print(f"XGBoost Balanced Accuracy: {balanced_acc:.3f}")
print(f"Number of classes: {len(label_encoder.classes_)}")

XGBoost Train Accuracy: 0.990
XGBoost Test Accuracy: 0.783
XGBoost Balanced Accuracy: 0.647
Number of classes: 24


In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Define the neural network
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.01,
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42
)

# Use the same preprocessing as previous models
mlp_pipeline = Pipeline([
    ("preprocessing", clone(preprocessing)),
    ("classifier", mlp)
])

# Train
mlp_pipeline.fit(X_train, y_train)

# Predictions
train_preds = mlp_pipeline.predict(X_train)
test_preds = mlp_pipeline.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_train, train_preds)
test_acc = accuracy_score(y_test, test_preds)
balanced_acc = balanced_accuracy_score(y_test, test_preds)

print(f"MLP Train Accuracy: {train_acc:.3f}")
print(f"MLP Test Accuracy: {test_acc:.3f}")
print(f"MLP Balanced Accuracy: {balanced_acc:.3f}")
print(f"Training iterations: {mlp_pipeline.named_steps['classifier'].n_iter_}")

MLP Train Accuracy: 0.941
MLP Test Accuracy: 0.783
MLP Balanced Accuracy: 0.647
Training iterations: 262
